# Analyze Dynamic Modular Charge Spreading

Loads script-generated sample/y0-averaged `dy_com(t)` products, adds zoomed `t <= 3` views, and keeps selected cycle-50 single-Hmod charge-density plus single-packet dy diagnostics for direct inspection.

Both the averaged and single-Hmod packet diagnostics include a wall-oriented chirality readout using `eta_wall * dy_com(t)`, with `eta_wall=+1` at `x=5` and `eta_wall=-1` at `x=11`.

The sample/y0 averaged section does not load raw covariance snapshots or construct modular Hamiltonians in the notebook. Generate those averaged dy products first with `colab_small_system_testing/src/run_sample_y0_averaged_dy_com.py`.


## Sample/Y0-Averaged Dy Configs

| Config | Nx | Ny | nshell | samples | y0 cuts | default cycle | output |
|---|---:|---:|---:|---:|---:|---:|---|
| `N16x30_nsh1_perfect_correction` | 16 | 30 | 1 | 10 | 30 | 50 | averaged dy + wall dy |
| `N16x30_nsh2_perfect_correction` | 16 | 30 | 2 | 10 | 30 | 50 | averaged dy + wall dy |
| `N16x40_nsh1_perfect_correction` | 16 | 40 | 1 | 10 | 40 | 50 | averaged dy + wall dy |
| `N16x40_nsh2_perfect_correction` | 16 | 40 | 2 | 10 | 40 | 50 | averaged dy + wall dy |


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

BUNDLE_NAME = 'colab_small_system_testing'

def is_bundle_root(path: Path) -> bool:
    return (path / 'src' / 'dynamic_modular_charge_spreading.py').exists() and (path / 'gpu_data').exists()

candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if is_bundle_root(candidate):
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME}')

SRC_DIR = BUNDLE_ROOT / 'src'
REPO_ROOT = BUNDLE_ROOT.parent
REPO_SRC_DIR = REPO_ROOT / 'src'
for path in (SRC_DIR, REPO_SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from dynamic_modular_charge_spreading import (
    eigensystem_from_h_mod,
    initial_occupancy_vector,
    modular_hamiltonian_from_restricted_covariance,
    reduced_site_index,
    restrict_covariance,
)
from fgtn.classA_U1FGTN import classA_U1FGTN

GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
HMOD_ROOT = GPU_DATA_ROOT / 'dynamic_modular_hamiltonians'
ANALYSIS_ROOT = BUNDLE_ROOT / 'analysis_outputs' / 'dynamic_modular_charge_spreading'
AVERAGED_DY_ROOT = ANALYSIS_ROOT / 'sample_y0_averaged_dy_com'
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)
DISPLAY_PLOTS_IN_NOTEBOOK = True

print(f'REPO_ROOT={REPO_ROOT}')
print(f'BUNDLE_ROOT={BUNDLE_ROOT}')
print(f'HMOD_ROOT={HMOD_ROOT}')
print(f'ANALYSIS_ROOT={ANALYSIS_ROOT}')
print(f'AVERAGED_DY_ROOT={AVERAGED_DY_ROOT}')


In [ ]:
def load_json(path: Path):
    with path.open('r', encoding='utf-8') as fh:
        return json.load(fh)

manifest_path = HMOD_ROOT / 'campaign_manifest.json'
manifest = load_json(manifest_path)
products = [Path(item['product_path']) for item in manifest['results']]
for product in products:
    if not product.exists():
        raise FileNotFoundError(product)
print(f'Loaded manifest: {manifest_path}')
print(f'Products: {len(products)}')
pd.read_csv(HMOD_ROOT / 'modular_hamiltonian_summary.csv').head()


## Sample/Y0-Averaged Dy Products

This section only loads outputs from `run_sample_y0_averaged_dy_com.py`. It does not load raw covariance snapshots or build modular Hamiltonians from Markov snapshots in the notebook.

`dy_com(t)` is the relative-y trajectory `y_com(t) - y_com(0)`, averaged over samples and cut offsets after each curve is computed.

The wall-oriented chirality curve is `eta_wall * dy_com(t)`, using `eta_wall=+1` for packets initialized at `x=5` and `eta_wall=-1` for packets initialized at `x=11`. Since `eta_wall` is constant for each packet, the averaged curve is `eta_wall * mean_{sample,y0} dy_com(t)`.

Each config subsection also includes an exact equilibrium domain-wall ground-state reference. The reference uses the same `Nx, Ny`, exact periodic DW covariance from `classA_U1FGTN.G_CI_domain_wall`, `alpha_1=1`, `alpha_2=30`, the `y0=0` half-window, and the same four packet initial conditions.


In [ ]:
AVERAGED_DY_CONFIG_KEYS = [
    'N16x30_nsh1_perfect_correction',
    'N16x30_nsh2_perfect_correction',
    'N16x40_nsh1_perfect_correction',
    'N16x40_nsh2_perfect_correction',
]
DY_ZOOM_T_MAX = 3.0
EXACT_REFERENCE_ALPHA_1 = 1.0
EXACT_REFERENCE_ALPHA_2 = 30.0
EXACT_REFERENCE_Y0 = 0
EXACT_REFERENCE_PERIODIC = True
EXACT_REFERENCE_TRIV_REGION_LOCAL_MODE = False
_EXACT_REFERENCE_CACHE = {}


def averaged_dy_product_path(source_key: str) -> Path:
    return AVERAGED_DY_ROOT / 'runs' / source_key / 'sample_y0_averaged_dy_com.npz'


def averaged_wall_eta(packet_x_dw) -> np.ndarray:
    packet_x_dw = np.asarray(packet_x_dw, dtype=np.int64)
    eta = np.empty(packet_x_dw.shape, dtype=np.float64)
    eta[packet_x_dw == 5] = 1.0
    eta[packet_x_dw == 11] = -1.0
    if not np.all(np.isin(packet_x_dw, [5, 11])):
        raise ValueError(f'Unexpected packet_x_dw values: {packet_x_dw}')
    return eta


def averaged_packet_metadata(ny: int):
    ny_sub = int(ny) // 2
    packet_labels = np.asarray(['x5_y0', f'x5_y{ny_sub - 1}', 'x11_y0', f'x11_y{ny_sub - 1}'])
    packet_x_dw = np.asarray([5, 5, 11, 11], dtype=np.int64)
    packet_y_rel = np.asarray([0, ny_sub - 1, 0, ny_sub - 1], dtype=np.int64)
    wall_eta = averaged_wall_eta(packet_x_dw)
    return packet_labels, packet_x_dw, packet_y_rel, wall_eta


def plot_averaged_curves(times, cycles, packet_labels, mean, sem, title, ylabel, *, time_max: float | None = None, wall_eta=None):
    times = np.asarray(times, dtype=np.float64)
    mean = np.asarray(mean, dtype=np.float64)
    sem = np.asarray(sem, dtype=np.float64)
    if time_max is None:
        time_mask = np.ones_like(times, dtype=bool)
        title_suffix = 'full time'
    else:
        time_mask = times <= float(time_max) + 1e-12
        title_suffix = f't <= {float(time_max):g}'
    plot_times = times[time_mask]
    fig, axes = plt.subplots(len(cycles), 1, figsize=(7.0, 3.5 * len(cycles)), constrained_layout=True, squeeze=False)
    for c_idx, cycle in enumerate(cycles):
        ax = axes[c_idx, 0]
        for p_idx, label in enumerate(packet_labels):
            y = mean[c_idx, p_idx, time_mask]
            err = sem[c_idx, p_idx, time_mask]
            curve_label = str(label) if wall_eta is None else f'{label}, eta={wall_eta[p_idx]:+.0f}'
            line = ax.plot(plot_times, y, lw=1.4, label=curve_label)[0]
            ax.fill_between(plot_times, y - err, y + err, color=line.get_color(), alpha=0.18, linewidth=0)
        ax.axhline(0.0, color='black', lw=0.8, alpha=0.5)
        ax.set_title(f'cycle {int(cycle)} | {title_suffix}')
        ax.set_xlabel('modular time')
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8, loc='best')
    fig.suptitle(f'{title} ({title_suffix})')
    if DISPLAY_PLOTS_IN_NOTEBOOK:
        display(fig)
    plt.close(fig)
    return fig


def plot_averaged_dy_product(path: Path):
    path = Path(path)
    if not path.exists():
        print(f'Missing averaged dy product: {path}')
        return None
    with np.load(path, allow_pickle=False) as data:
        metadata = json.loads(str(data['metadata_json']))
        source_key = metadata['source_key']
        cycles = np.asarray(data['snapshot_cycles'], dtype=np.int64)
        times = np.asarray(data['times'], dtype=np.float64)
        packet_labels = [str(label) for label in data['packet_labels']]
        packet_x_dw = np.asarray(data['packet_x_dw'], dtype=np.int64)
        wall_eta = np.asarray(data['wall_eta'], dtype=np.float64) if 'wall_eta' in data else averaged_wall_eta(packet_x_dw)
        dy_mean = np.asarray(data['dy_com_mean'], dtype=np.float64)
        dy_sem = np.asarray(data['dy_com_sem'], dtype=np.float64)
        signed_mean = np.asarray(data['signed_tangent_dy_mean'], dtype=np.float64) if 'signed_tangent_dy_mean' in data else wall_eta[None, :, None] * dy_mean
        signed_sem = np.asarray(data['signed_tangent_dy_sem'], dtype=np.float64) if 'signed_tangent_dy_sem' in data else dy_sem
    plot_averaged_curves(
        times,
        cycles,
        packet_labels,
        dy_mean,
        dy_sem,
        f'{source_key}: sample/y0-averaged dy trajectory',
        r'$\mathrm{mean}_{sample,y0}\,\Delta\langle y_{rel}\rangle$',
    )
    plot_averaged_curves(
        times,
        cycles,
        packet_labels,
        dy_mean,
        dy_sem,
        f'{source_key}: sample/y0-averaged dy trajectory',
        r'$\mathrm{mean}_{sample,y0}\,\Delta\langle y_{rel}\rangle$',
        time_max=DY_ZOOM_T_MAX,
    )
    plot_averaged_curves(
        times,
        cycles,
        packet_labels,
        signed_mean,
        signed_sem,
        f'{source_key}: sample/y0-averaged wall-oriented dy trajectory',
        r'$\mathrm{mean}_{sample,y0}\,\eta_{wall}\Delta\langle y_{rel}\rangle$',
        wall_eta=wall_eta,
    )
    plot_averaged_curves(
        times,
        cycles,
        packet_labels,
        signed_mean,
        signed_sem,
        f'{source_key}: sample/y0-averaged wall-oriented dy trajectory',
        r'$\mathrm{mean}_{sample,y0}\,\eta_{wall}\Delta\langle y_{rel}\rangle$',
        time_max=DY_ZOOM_T_MAX,
        wall_eta=wall_eta,
    )
    return path


def reference_time_grid(nx: int, dt: float = 0.01, t_final: float | None = None):
    if t_final is None:
        t_final = 2.0 * int(nx)
    n_steps = int(round(float(t_final) / float(dt)))
    if not np.isclose(n_steps * float(dt), float(t_final)):
        raise ValueError('dt does not divide t_final for exact reference.')
    return float(dt) * np.arange(n_steps + 1, dtype=np.float64)


def reference_packet_occupancy(nx: int, ny: int, x_dw: int, y_rel: int):
    ny_sub = int(ny) // 2
    q0 = np.zeros(int(nx) * ny_sub * 2, dtype=np.float64)
    for orbital in (0, 1):
        q0[reduced_site_index(nx, int(x_dw) % int(nx), int(y_rel), orbital)] = 1.0
    if not np.isclose(float(q0.sum()), 2.0):
        raise ValueError('Exact reference packet occupancy should have charge 2.')
    return q0


def reference_periodic_x_window(nx: int, x_center: int, radius: int = 2):
    xs = np.arange(int(nx), dtype=np.int64)
    forward = (xs - int(x_center)) % int(nx)
    backward = (int(x_center) - xs) % int(nx)
    return np.minimum(forward, backward) <= int(radius)


def reference_evolve_charge(h_vals, h_vecs, q0, nx: int, ny: int, times, time_chunk: int = 32):
    h_vals = np.asarray(h_vals, dtype=np.float64)
    h_vecs = np.asarray(h_vecs, dtype=np.complex128)
    q0 = np.asarray(q0, dtype=np.float64)
    ny_sub = int(ny) // 2
    occ = np.flatnonzero(q0 > 0.5)
    occ_coeff = h_vecs[occ, :].conj().T
    N_xy = np.empty((len(times), int(nx), ny_sub), dtype=np.float32)
    charges = np.empty((len(times),), dtype=np.float64)
    for start in range(0, len(times), int(time_chunk)):
        stop = min(len(times), start + int(time_chunk))
        phase = np.exp(-1j * times[start:stop, None] * h_vals[None, :])
        amplitudes = np.einsum('ik,tk,ka->tia', h_vecs, phase, occ_coeff, optimize=True)
        orbital_density = np.sum(np.abs(amplitudes) ** 2, axis=2).real
        maps = orbital_density.reshape(stop - start, ny_sub, int(nx), 2).sum(axis=-1).transpose(0, 2, 1)
        N_xy[start:stop] = maps.astype(np.float32, copy=False)
        charges[start:stop] = maps.sum(axis=(1, 2), dtype=np.float64)
    return N_xy, charges


def reference_dy_from_maps(N_xy, nx: int, ny: int, x_dw: int, x_window_radius: int = 2):
    N_xy = np.asarray(N_xy, dtype=np.float64)
    ny_sub = int(ny) // 2
    x_mask = reference_periodic_x_window(nx, x_dw, x_window_radius)
    y_values = np.arange(ny_sub, dtype=np.float64)
    window_density = N_xy[:, x_mask, :]
    Q_window = window_density.sum(axis=(1, 2), dtype=np.float64)
    if np.any(Q_window <= 1e-12):
        raise ValueError('Exact reference window charge became too small.')
    y_com = (window_density * y_values[None, None, :]).sum(axis=(1, 2), dtype=np.float64) / Q_window
    return y_com - y_com[0], y_com, Q_window


def exact_equilibrium_reference_curves(nx: int, ny: int):
    key = (int(nx), int(ny), EXACT_REFERENCE_ALPHA_1, EXACT_REFERENCE_ALPHA_2, EXACT_REFERENCE_Y0)
    if key in _EXACT_REFERENCE_CACHE:
        return _EXACT_REFERENCE_CACHE[key]

    model = classA_U1FGTN(
        int(nx),
        int(ny),
        nshell=None,
        DW=True,
        alpha_1=EXACT_REFERENCE_ALPHA_1,
        alpha_2=EXACT_REFERENCE_ALPHA_2,
    )
    G_eq = model.G_CI_domain_wall(
        periodic=EXACT_REFERENCE_PERIODIC,
        triv_region_local_mode=EXACT_REFERENCE_TRIV_REGION_LOCAL_MODE,
    )
    G_A = restrict_covariance(G_eq, nx=int(nx), ny=int(ny), y0=EXACT_REFERENCE_Y0)
    mod = modular_hamiltonian_from_restricted_covariance(G_A, eps=1e-10)
    eig = eigensystem_from_h_mod(mod['h_mod'])
    times = reference_time_grid(nx)
    packet_labels, packet_x_dw, packet_y_rel, wall_eta = averaged_packet_metadata(ny)

    dy_com = np.empty((len(packet_labels), len(times)), dtype=np.float64)
    y_com = np.empty_like(dy_com)
    Q_window = np.empty_like(dy_com)
    charges = np.empty_like(dy_com)
    for p_idx, (x_dw, y_rel) in enumerate(zip(packet_x_dw, packet_y_rel)):
        q0 = reference_packet_occupancy(nx, ny, int(x_dw), int(y_rel))
        N_xy, charge = reference_evolve_charge(eig['h_vals'], eig['h_vecs'], q0, nx, ny, times)
        dy, y, q = reference_dy_from_maps(N_xy, nx, ny, int(x_dw))
        dy_com[p_idx] = dy
        y_com[p_idx] = y
        Q_window[p_idx] = q
        charges[p_idx] = charge

    result = {
        'times': times,
        'packet_labels': packet_labels,
        'packet_x_dw': packet_x_dw,
        'packet_y_rel': packet_y_rel,
        'wall_eta': wall_eta,
        'dy_com': dy_com,
        'signed_tangent_dy': wall_eta[:, None] * dy_com,
        'y_com': y_com,
        'Q_window': Q_window,
        'charges': charges,
        'dw_loc': np.asarray(getattr(model, 'DW_loc', []), dtype=np.int64),
        'h_mod_hermiticity_error': float(eig['h_mod_hermiticity_error']),
        'h_reconstruction_error': float(eig['reconstruction_error']),
        'covariance_hermiticity_error': float(mod['covariance_hermiticity_error']),
        'clip_low_count': int(mod['clip_low_count']),
        'clip_high_count': int(mod['clip_high_count']),
    }
    _EXACT_REFERENCE_CACHE[key] = result
    return result


def plot_exact_equilibrium_reference_config(source_key: str, nx: int, ny: int):
    ref = exact_equilibrium_reference_curves(nx, ny)
    print(
        f"Exact reference {source_key}: DW_loc={ref['dw_loc'].tolist()}, "
        f"H herm err={ref['h_mod_hermiticity_error']:.3e}, "
        f"recon err={ref['h_reconstruction_error']:.3e}, "
        f"clip=({ref['clip_low_count']}, {ref['clip_high_count']})"
    )
    max_charge_drift = float(np.max(np.abs(ref['charges'] - ref['charges'][:, [0]])))
    print(f'Exact reference max packet charge drift: {max_charge_drift:.3e}')
    cycles = np.asarray([50], dtype=np.int64)
    dy = ref['dy_com'][None, :, :]
    signed = ref['signed_tangent_dy'][None, :, :]
    zero_sem = np.zeros_like(dy)
    plot_averaged_curves(
        ref['times'],
        cycles,
        ref['packet_labels'],
        dy,
        zero_sem,
        f'{source_key}: exact equilibrium DW GS reference dy trajectory',
        r'$\Delta\langle y_{rel}\rangle$',
    )
    plot_averaged_curves(
        ref['times'],
        cycles,
        ref['packet_labels'],
        dy,
        zero_sem,
        f'{source_key}: exact equilibrium DW GS reference dy trajectory',
        r'$\Delta\langle y_{rel}\rangle$',
        time_max=DY_ZOOM_T_MAX,
    )
    plot_averaged_curves(
        ref['times'],
        cycles,
        ref['packet_labels'],
        signed,
        zero_sem,
        f'{source_key}: exact equilibrium DW GS reference wall-oriented dy trajectory',
        r'$\eta_{wall}\Delta\langle y_{rel}\rangle$',
        wall_eta=ref['wall_eta'],
    )
    plot_averaged_curves(
        ref['times'],
        cycles,
        ref['packet_labels'],
        signed,
        zero_sem,
        f'{source_key}: exact equilibrium DW GS reference wall-oriented dy trajectory',
        r'$\eta_{wall}\Delta\langle y_{rel}\rangle$',
        time_max=DY_ZOOM_T_MAX,
        wall_eta=ref['wall_eta'],
    )
    return ref


def plot_averaged_dy_config(source_key: str):
    if source_key not in AVERAGED_DY_CONFIG_KEYS:
        raise KeyError(f'Unknown averaged dy config: {source_key}')
    return plot_averaged_dy_product(averaged_dy_product_path(source_key))


def display_averaged_dy_summary():
    summary_path = AVERAGED_DY_ROOT / 'sample_y0_averaged_dy_com_summary.csv'
    if summary_path.exists():
        display(pd.read_csv(summary_path))
    else:
        print(f'Missing averaged dy summary: {summary_path}')


### N16x30_nsh1_perfect_correction

`Nx=16`, `Ny=30`, `nshell=1`, default cycle `50`.

Displays raw sample/y0-averaged `dy_com(t)`, the `t <= 3` zoom, wall-oriented `eta_wall * dy_com(t)`, and its `t <= 3` zoom.

Exact reference: exact periodic equilibrium DW ground state with `alpha_1=1`, `alpha_2=30`, `y0=0`.


In [ ]:
plot_averaged_dy_config('N16x30_nsh1_perfect_correction')
exact_ref_N16x30_nsh1 = plot_exact_equilibrium_reference_config('N16x30_nsh1_perfect_correction', nx=16, ny=30)


### N16x30_nsh2_perfect_correction

`Nx=16`, `Ny=30`, `nshell=2`, default cycle `50`.

Displays raw sample/y0-averaged `dy_com(t)`, the `t <= 3` zoom, wall-oriented `eta_wall * dy_com(t)`, and its `t <= 3` zoom.

Exact reference: exact periodic equilibrium DW ground state with `alpha_1=1`, `alpha_2=30`, `y0=0`.


In [ ]:
plot_averaged_dy_config('N16x30_nsh2_perfect_correction')
exact_ref_N16x30_nsh2 = plot_exact_equilibrium_reference_config('N16x30_nsh2_perfect_correction', nx=16, ny=30)


### N16x40_nsh1_perfect_correction

`Nx=16`, `Ny=40`, `nshell=1`, default cycle `50`.

Displays raw sample/y0-averaged `dy_com(t)`, the `t <= 3` zoom, wall-oriented `eta_wall * dy_com(t)`, and its `t <= 3` zoom.

Exact reference: exact periodic equilibrium DW ground state with `alpha_1=1`, `alpha_2=30`, `y0=0`.


In [ ]:
plot_averaged_dy_config('N16x40_nsh1_perfect_correction')
exact_ref_N16x40_nsh1 = plot_exact_equilibrium_reference_config('N16x40_nsh1_perfect_correction', nx=16, ny=40)


### N16x40_nsh2_perfect_correction

`Nx=16`, `Ny=40`, `nshell=2`, default cycle `50`.

Displays raw sample/y0-averaged `dy_com(t)`, the `t <= 3` zoom, wall-oriented `eta_wall * dy_com(t)`, and its `t <= 3` zoom.

Exact reference: exact periodic equilibrium DW ground state with `alpha_1=1`, `alpha_2=30`, `y0=0`.


In [ ]:
plot_averaged_dy_config('N16x40_nsh2_perfect_correction')
exact_ref_N16x40_nsh2 = plot_exact_equilibrium_reference_config('N16x40_nsh2_perfect_correction', nx=16, ny=40)


### Sample/Y0-Averaged Summary

Compact table written by `run_sample_y0_averaged_dy_com.py`, if available.


In [ ]:
display_averaged_dy_summary()


## Heisenberg Charge-Evolution Helpers


In [ ]:
DT = 0.01
T_FINAL_FACTOR = 2
TIME_CHUNK = 32
X_WINDOW_RADIUS = 2
PACKET_X_DW = (5, 5, 11, 11)
WALL_ORIENTATION = {5: 1.0, 11: -1.0}


def time_grid(nx: int, dt: float = DT, t_final=None):
    if t_final is None:
        t_final = T_FINAL_FACTOR * int(nx)
    n_steps = int(round(float(t_final) / float(dt)))
    if not np.isclose(n_steps * float(dt), float(t_final)):
        raise ValueError('dt does not divide t_final')
    return float(dt) * np.arange(n_steps + 1, dtype=np.float64)


def wall_orientation_eta(x_dw: int) -> float:
    x_dw = int(x_dw)
    if x_dw not in WALL_ORIENTATION:
        raise ValueError(f'No wall orientation convention for x_dw={x_dw}')
    return float(WALL_ORIENTATION[x_dw])


def packet_specs(ny: int):
    ny_sub = int(ny) // 2
    specs = [
        {'label': 'x5_y0', 'x_dw': 5, 'y_rel': 0},
        {'label': f'x5_y{ny_sub - 1}', 'x_dw': 5, 'y_rel': ny_sub - 1},
        {'label': 'x11_y0', 'x_dw': 11, 'y_rel': 0},
        {'label': f'x11_y{ny_sub - 1}', 'x_dw': 11, 'y_rel': ny_sub - 1},
    ]
    for spec in specs:
        spec['wall_eta'] = wall_orientation_eta(spec['x_dw'])
    return specs


def single_packet_occupancy_vector(nx: int, ny: int, x_dw: int, y_rel: int) -> np.ndarray:
    nx = int(nx)
    ny = int(ny)
    ny_sub = ny // 2
    q0 = np.zeros(nx * ny_sub * 2, dtype=np.float64)
    for orbital in (0, 1):
        q0[reduced_site_index(nx, int(x_dw) % nx, int(y_rel), orbital)] = 1.0
    if not np.isclose(float(q0.sum()), 2.0):
        raise ValueError(f'Single-packet occupancy should contain charge 2, got {q0.sum()}.')
    if np.max(np.abs(q0 * q0 - q0)) > 0.0:
        raise ValueError('Initial occupancy must be idempotent.')
    return q0


def periodic_x_window(nx: int, x_center: int, radius: int = X_WINDOW_RADIUS) -> np.ndarray:
    xs = np.arange(int(nx), dtype=np.int64)
    forward = (xs - int(x_center)) % int(nx)
    backward = (int(x_center) - xs) % int(nx)
    distance = np.minimum(forward, backward)
    return distance <= int(radius)


def dy_from_charge_maps(N_xy, times, nx: int, ny: int, x_dw: int, *, x_window_radius: int = X_WINDOW_RADIUS):
    N_xy = np.asarray(N_xy, dtype=np.float64)
    ny_sub = int(ny) // 2
    x_mask = periodic_x_window(nx, x_dw, x_window_radius)
    y_values = np.arange(ny_sub, dtype=np.float64)
    window_density = N_xy[:, x_mask, :]
    Q_window = window_density.sum(axis=(1, 2), dtype=np.float64)
    if np.any(~np.isfinite(Q_window)) or float(np.min(Q_window)) <= 1e-12:
        raise ValueError('Window charge is not finite and positive.')
    y_com = (window_density * y_values[None, None, :]).sum(axis=(1, 2), dtype=np.float64) / Q_window
    dy_com = y_com - y_com[0]
    if not (np.all(np.isfinite(y_com)) and np.all(np.isfinite(dy_com))):
        raise ValueError('dy trajectory contains non-finite values.')
    return {
        'dy_com': dy_com,
        'y_com': y_com,
        'Q_window': Q_window,
        'min_Q_window': float(np.min(Q_window)),
    }


def evolve_charge_from_eigensystem(h_vals, h_vecs, q0, nx, ny, times, time_chunk=TIME_CHUNK):
    h_vals = np.asarray(h_vals, dtype=np.float64)
    h_vecs = np.asarray(h_vecs, dtype=np.complex128)
    q0 = np.asarray(q0, dtype=np.float64)
    ny_sub = int(ny) // 2
    occ = np.flatnonzero(q0 > 0.5)
    occ_coeff = h_vecs[occ, :].conj().T
    N_xy = np.empty((len(times), int(nx), ny_sub), dtype=np.float32)
    charges = np.empty((len(times),), dtype=np.float64)
    for start in range(0, len(times), int(time_chunk)):
        stop = min(len(times), start + int(time_chunk))
        phase = np.exp(-1j * times[start:stop, None] * h_vals[None, :])
        amplitudes = np.einsum('ik,tk,ka->tia', h_vecs, phase, occ_coeff, optimize=True)
        orbital_density = np.sum(np.abs(amplitudes) ** 2, axis=2).real
        maps = orbital_density.reshape(stop - start, ny_sub, int(nx), 2).sum(axis=-1).transpose(0, 2, 1)
        N_xy[start:stop] = maps.astype(np.float32, copy=False)
        charges[start:stop] = maps.sum(axis=(1, 2), dtype=np.float64)
    return {
        'N_xy': N_xy,
        'charges': charges,
        'max_charge_drift': float(np.max(np.abs(charges - charges[0]))),
    }


def single_packet_dy_curves_from_eigensystem(h_vals, h_vecs, nx: int, ny: int, times):
    specs = packet_specs(ny)
    wall_eta = np.asarray([spec['wall_eta'] for spec in specs], dtype=np.float64)
    dy_com = np.empty((len(specs), len(times)), dtype=np.float64)
    y_com = np.empty_like(dy_com)
    Q_window = np.empty_like(dy_com)
    total_charge = np.empty_like(dy_com)
    records = []
    for p_idx, spec in enumerate(specs):
        q0 = single_packet_occupancy_vector(nx, ny, spec['x_dw'], spec['y_rel'])
        result = evolve_charge_from_eigensystem(h_vals, h_vecs, q0, nx, ny, times)
        dy_result = dy_from_charge_maps(result['N_xy'], times, nx, ny, spec['x_dw'])
        dy_com[p_idx] = dy_result['dy_com']
        y_com[p_idx] = dy_result['y_com']
        Q_window[p_idx] = dy_result['Q_window']
        total_charge[p_idx] = result['charges']
        signed_tangent = wall_eta[p_idx] * dy_com[p_idx]
        records.append({
            'packet_label': spec['label'],
            'packet_x_dw': int(spec['x_dw']),
            'packet_y_rel': int(spec['y_rel']),
            'wall_eta': float(wall_eta[p_idx]),
            'initial_charge': float(result['charges'][0]),
            'max_charge_drift': result['max_charge_drift'],
            'final_dy_com': float(dy_com[p_idx, -1]),
            'max_abs_dy_com': float(np.max(np.abs(dy_com[p_idx]))),
            'final_signed_tangent_dy': float(signed_tangent[-1]),
            'max_abs_signed_tangent_dy': float(np.max(np.abs(signed_tangent))),
            'min_Q_window': float(dy_result['min_Q_window']),
        })
    signed_tangent_dy = wall_eta[:, None] * dy_com
    return {
        'packet_labels': np.asarray([spec['label'] for spec in specs]),
        'packet_x_dw': np.asarray([spec['x_dw'] for spec in specs], dtype=np.int64),
        'packet_y_rel': np.asarray([spec['y_rel'] for spec in specs], dtype=np.int64),
        'wall_eta': wall_eta,
        'dy_com': dy_com,
        'signed_tangent_dy': signed_tangent_dy,
        'y_com': y_com,
        'Q_window': Q_window,
        'total_charge': total_charge,
        'records': records,
    }


def save_npz(path: Path, **arrays):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **arrays)


def plot_dy_curves(times, dy_com, packet_labels, title, path, *, time_max: float | None = None):
    times = np.asarray(times, dtype=np.float64)
    dy_com = np.asarray(dy_com, dtype=np.float64)
    if time_max is None:
        time_mask = np.ones_like(times, dtype=bool)
        suffix = 'full time'
    else:
        time_mask = times <= float(time_max) + 1e-12
        suffix = f't <= {float(time_max):g}'
    fig, ax = plt.subplots(figsize=(7.0, 3.6), constrained_layout=True)
    for p_idx, label in enumerate(packet_labels):
        ax.plot(times[time_mask], dy_com[p_idx, time_mask], lw=1.5, label=str(label))
    ax.axhline(0.0, color='black', lw=0.8, alpha=0.5)
    ax.set_title(f'{title} ({suffix})')
    ax.set_xlabel('modular time')
    ax.set_ylabel(r'$\Delta\langle y_{rel}\rangle$')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, loc='best')
    fig.savefig(path, dpi=180, bbox_inches='tight')
    if DISPLAY_PLOTS_IN_NOTEBOOK:
        display(fig)
    plt.close(fig)


def plot_signed_tangent_dy_curves(times, signed_tangent_dy, packet_labels, wall_eta, title, path, *, time_max: float | None = None):
    times = np.asarray(times, dtype=np.float64)
    signed_tangent_dy = np.asarray(signed_tangent_dy, dtype=np.float64)
    wall_eta = np.asarray(wall_eta, dtype=np.float64)
    if time_max is None:
        time_mask = np.ones_like(times, dtype=bool)
        suffix = 'full time'
    else:
        time_mask = times <= float(time_max) + 1e-12
        suffix = f't <= {float(time_max):g}'
    fig, ax = plt.subplots(figsize=(7.0, 3.6), constrained_layout=True)
    for p_idx, label in enumerate(packet_labels):
        curve_label = f'{label}, eta={wall_eta[p_idx]:+.0f}'
        ax.plot(times[time_mask], signed_tangent_dy[p_idx, time_mask], lw=1.5, label=curve_label)
    ax.axhline(0.0, color='black', lw=0.8, alpha=0.5)
    ax.set_title(f'{title}: wall-oriented tangent dy ({suffix})')
    ax.set_xlabel('modular time')
    ax.set_ylabel(r'$\eta_{wall}\Delta\langle y_{rel}\rangle$')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, loc='best')
    fig.savefig(path, dpi=180, bbox_inches='tight')
    if DISPLAY_PLOTS_IN_NOTEBOOK:
        display(fig)
    plt.close(fig)


def plot_panels(N_xy, times, cycles, title, path):
    target_times = [0, 8, 16, 32]
    t_indices = [int(np.argmin(np.abs(times - t))) for t in target_times]
    vmax = max(1e-12, float(np.max(N_xy[:, t_indices])))
    fig, axes = plt.subplots(len(cycles), len(t_indices), figsize=(3.2 * len(t_indices), 2.6 * len(cycles)), constrained_layout=True, squeeze=False)
    for r, cycle in enumerate(cycles):
        for c, t_idx in enumerate(t_indices):
            ax = axes[r, c]
            im = ax.imshow(N_xy[r, t_idx].T, origin='lower', aspect='auto', cmap='viridis', vmin=0, vmax=vmax)
            ax.set_title(f'cycle {int(cycle)}, t={times[t_idx]:.1f}')
            ax.set_xlabel('x')
            ax.set_ylabel('y_rel')
            for x_dw in [5, 11]:
                ax.axvline(float(x_dw), color='white', lw=0.8, alpha=0.8)
    fig.suptitle(title)
    fig.colorbar(im, ax=axes.ravel().tolist(), label='cell charge')
    fig.savefig(path, dpi=180, bbox_inches='tight')
    if DISPLAY_PLOTS_IN_NOTEBOOK:
        display(fig)
    plt.close(fig)


## Single-Hmod Diagnostics: Selected Cycle-50 Charge Density And Dy Curves

This section uses precomputed modular Hamiltonians. The single-packet dy curves use only `single_y0_sample`, i.e. the modular Hamiltonian from `sample_idx=0` and `y0=0`.

The chirality readout is `eta_wall * dy_com(t)`, with `eta_wall=+1` for the `x=5` wall and `eta_wall=-1` for the `x=11` wall. If this sign-corrected trajectory lines up across the two walls, it supports counterpropagating wall-oriented motion rather than simple edge-inward breathing.


In [ ]:
TARGET_CYCLE = 50
SEQUENCE_CONFIG_KEYS = [
    'N16x30_nsh1_perfect_correction',
    'N16x40_nsh1_perfect_correction',
]
CONSTRUCTIONS = [
    ('single', 'single_y0_sample'),
    ('avg', 'sample_y0_avg_hmod'),
]
SELECTED_ROOT = ANALYSIS_ROOT / f'selected_cycle{TARGET_CYCLE}'
SELECTED_ROOT.mkdir(parents=True, exist_ok=True)

product_by_key = {item['source_key']: Path(item['product_path']) for item in manifest['results']}
missing = [key for key in SEQUENCE_CONFIG_KEYS if key not in product_by_key]
if missing:
    raise KeyError(f'SEQUENCE_CONFIG_KEYS not found in manifest: {missing}')


def cycle_index_for(cycles, target_cycle: int = TARGET_CYCLE):
    matches = np.flatnonzero(np.asarray(cycles, dtype=np.int64) == int(target_cycle))
    if len(matches) != 1:
        raise ValueError(f'cycle {target_cycle} not found exactly once in {list(cycles)}')
    return int(matches[0])


def run_combined_charge_cycle(data, metadata, prefix, construction, cycle_idx, cycle, out_dir):
    nx = int(data['Nx'])
    ny = int(data['Ny'])
    times = time_grid(nx)
    q0 = initial_occupancy_vector(nx, ny)
    result = evolve_charge_from_eigensystem(data[f'{prefix}_h_vals'][cycle_idx], data[f'{prefix}_h_vecs'][cycle_idx], q0, nx, ny, times)
    N_xy = result['N_xy'][None, ...]
    charges = result['charges'][None, ...]
    cycles = np.asarray([cycle], dtype=np.int64)
    out_path = out_dir / f'{construction}_cycle{cycle}_charge_spreading.npz'
    save_npz(
        out_path,
        N_xy=N_xy,
        charges=charges,
        times=times,
        snapshot_cycles=cycles,
        initial_occupancy=q0,
        metadata_json=json.dumps({**metadata, 'construction': construction, 'cycle': int(cycle)}, sort_keys=True),
    )
    fig_path = out_dir / f'{construction}_cycle{cycle}_charge_panels.png'
    plot_panels(N_xy, times, cycles, f"{metadata['source_key']} | {construction} | cycle {cycle}", fig_path)
    return {
        'source_key': metadata['source_key'],
        'Nx': nx,
        'Ny': ny,
        'nshell': int(data['nshell']),
        'construction': construction,
        'cycle': int(cycle),
        'initial_charge': float(charges[0, 0]),
        'max_charge_drift': result['max_charge_drift'],
        'npz_path': str(out_path),
        'figure_path': str(fig_path),
    }


def run_single_y0_sample_packet_dy_cycle(data, metadata, cycle_idx, cycle, out_dir):
    nx = int(data['Nx'])
    ny = int(data['Ny'])
    times = time_grid(nx)
    result = single_packet_dy_curves_from_eigensystem(data['single_h_vals'][cycle_idx], data['single_h_vecs'][cycle_idx], nx, ny, times)
    out_path = out_dir / f'single_y0_sample_cycle{cycle}_single_packet_dy.npz'
    save_npz(
        out_path,
        times=times,
        snapshot_cycles=np.asarray([cycle], dtype=np.int64),
        packet_labels=result['packet_labels'],
        packet_x_dw=result['packet_x_dw'],
        packet_y_rel=result['packet_y_rel'],
        wall_eta=result['wall_eta'],
        dy_com=result['dy_com'],
        signed_tangent_dy=result['signed_tangent_dy'],
        y_com=result['y_com'],
        Q_window=result['Q_window'],
        total_charge=result['total_charge'],
        metadata_json=json.dumps({**metadata, 'construction': 'single_y0_sample', 'cycle': int(cycle), 'sample_idx': 0, 'y0': 0}, sort_keys=True),
    )
    fig_path = out_dir / f'single_y0_sample_cycle{cycle}_single_packet_dy.png'
    zoom_fig_path = out_dir / f'single_y0_sample_cycle{cycle}_single_packet_dy_zoom_t3.png'
    signed_fig_path = out_dir / f'single_y0_sample_cycle{cycle}_signed_tangent_dy.png'
    signed_zoom_fig_path = out_dir / f'single_y0_sample_cycle{cycle}_signed_tangent_dy_zoom_t3.png'
    title = f"{metadata['source_key']} | single_y0_sample | cycle {cycle}"
    plot_dy_curves(times, result['dy_com'], result['packet_labels'], title, fig_path)
    plot_dy_curves(times, result['dy_com'], result['packet_labels'], title, zoom_fig_path, time_max=DY_ZOOM_T_MAX)
    plot_signed_tangent_dy_curves(times, result['signed_tangent_dy'], result['packet_labels'], result['wall_eta'], title, signed_fig_path)
    plot_signed_tangent_dy_curves(times, result['signed_tangent_dy'], result['packet_labels'], result['wall_eta'], title, signed_zoom_fig_path, time_max=DY_ZOOM_T_MAX)
    records = []
    for record in result['records']:
        records.append({
            'source_key': metadata['source_key'],
            'Nx': nx,
            'Ny': ny,
            'nshell': int(data['nshell']),
            'construction': 'single_y0_sample',
            'cycle': int(cycle),
            'sample_idx': 0,
            'y0': 0,
            **record,
            'npz_path': str(out_path),
            'figure_path': str(fig_path),
            'zoom_figure_path': str(zoom_fig_path),
            'signed_tangent_figure_path': str(signed_fig_path),
            'signed_tangent_zoom_figure_path': str(signed_zoom_fig_path),
        })
    return records


def run_config_sequence(source_key: str):
    product = product_by_key[source_key]
    charge_records = []
    dy_records = []
    with np.load(product, allow_pickle=False) as data:
        metadata = json.loads(str(data['metadata_json']))
        cycles = np.asarray(data['snapshot_cycles'], dtype=np.int64)
        cycle_idx = cycle_index_for(cycles, TARGET_CYCLE)
        cycle = int(cycles[cycle_idx])
        out_dir = SELECTED_ROOT / source_key
        print(f'Running {source_key}: cycle={cycle}, output={out_dir}')
        for prefix, construction in CONSTRUCTIONS:
            print(f'  {construction} charge panels')
            charge_records.append(run_combined_charge_cycle(data, metadata, prefix, construction, cycle_idx, cycle, out_dir))
        print('  single_y0_sample single-packet dy and wall-oriented tangent curves')
        dy_records.extend(run_single_y0_sample_packet_dy_cycle(data, metadata, cycle_idx, cycle, out_dir))
    charge_df = pd.DataFrame(charge_records)
    dy_df = pd.DataFrame(dy_records)
    display(charge_df)
    display(dy_df)
    return charge_records, dy_records


print(f'TARGET_CYCLE={TARGET_CYCLE}')
print('Sequence configs:')
for key in SEQUENCE_CONFIG_KEYS:
    print(f'  {key}')


## Single-Hmod Diagnostic Sequence 1: N16x30_nsh1_perfect_correction, Cycle 50

Runs the selected cycle-50 charge-density panels for both `single_y0_sample` and `sample_y0_avg_hmod`, then plots four single-packet dy trajectories using `single_y0_sample` only.

Both raw `dy_com(t)` and wall-oriented `eta_wall * dy_com(t)` are shown, with full-time and `t <= 3` zoom plots.


In [ ]:
sequence1_charge_records, sequence1_dy_records = run_config_sequence(SEQUENCE_CONFIG_KEYS[0])


## Single-Hmod Diagnostic Sequence 2: N16x40_nsh1_perfect_correction, Cycle 50

Runs the selected cycle-50 charge-density panels for both `single_y0_sample` and `sample_y0_avg_hmod`, then plots four single-packet dy trajectories using `single_y0_sample` only.

Both raw `dy_com(t)` and wall-oriented `eta_wall * dy_com(t)` are shown, with full-time and `t <= 3` zoom plots.


In [ ]:
sequence2_charge_records, sequence2_dy_records = run_config_sequence(SEQUENCE_CONFIG_KEYS[1])


## Single-Hmod Diagnostic Cycle-50 Summary

Collects the two selected config charge-density sequences and the `single_y0_sample` single-packet dy/chirality trajectories into compact CSV/parquet summaries.


In [ ]:
selected_charge_records = sequence1_charge_records + sequence2_charge_records
selected_dy_records = sequence1_dy_records + sequence2_dy_records

selected_charge_df = pd.DataFrame(selected_charge_records)
selected_dy_df = pd.DataFrame(selected_dy_records)

charge_csv = SELECTED_ROOT / f'selected_cycle{TARGET_CYCLE}_charge_spreading_summary.csv'
selected_charge_df.to_csv(charge_csv, index=False)
try:
    selected_charge_df.to_parquet(SELECTED_ROOT / f'selected_cycle{TARGET_CYCLE}_charge_spreading_summary.parquet', index=False)
except Exception as exc:
    print(f'charge parquet skipped: {type(exc).__name__}: {exc}')

dy_csv = SELECTED_ROOT / f'selected_cycle{TARGET_CYCLE}_single_y0_sample_packet_dy_summary.csv'
selected_dy_df.to_csv(dy_csv, index=False)
try:
    selected_dy_df.to_parquet(SELECTED_ROOT / f'selected_cycle{TARGET_CYCLE}_single_y0_sample_packet_dy_summary.parquet', index=False)
except Exception as exc:
    print(f'dy parquet skipped: {type(exc).__name__}: {exc}')

expected_charge_rows = len(SEQUENCE_CONFIG_KEYS) * len(CONSTRUCTIONS)
if len(selected_charge_df) != expected_charge_rows:
    raise ValueError(f'charge rows {len(selected_charge_df)} != expected {expected_charge_rows}')
expected_dy_rows = len(SEQUENCE_CONFIG_KEYS) * len(packet_specs(30))
if len(selected_dy_df) != expected_dy_rows:
    raise ValueError(f'dy rows {len(selected_dy_df)} != expected {expected_dy_rows}')

print(f'Saved charge summary: {charge_csv}')
print(f'Saved single_y0_sample packet dy summary: {dy_csv}')
display(selected_charge_df)
display(selected_dy_df)
